# Interpolation Case 02: Interpolation Denoising

This notebook follows the myofibroblast interpolation and denoising case study in the Navigo study, focusing on whether Navigo corrects an anomalous myofibroblast proportion at E18.25 and restores coherent marker and pathway readouts along the muscle trajectory.

Biological objective:
1. Generate adjacent-time interpolation predictions for the muscle trajectory and recover the denoised myofibroblast population.
2. Compare noisy versus Navigo-derived marker dynamics across late developmental stages.
3. Test whether the denoised population recovers biologically relevant pathway enrichment at E18.25.


Import required packages and set deterministic seeds.

In [ ]:
import configparser
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from scipy import sparse, stats
from sklearn.neighbors import KNeighborsClassifier

Define shared helpers for path resolution, dense conversion, and DEG testing.

In [ ]:
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'docs' / 'tutorials').exists() and (p / 'navigo').exists():
            return p
    raise RuntimeError('Could not locate repository root containing docs/tutorials and the navigo package.')


def to_dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    n = pvals.size
    order = np.argsort(pvals)
    ranked = pvals[order]

    adj_ranked = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = min(prev, ranked[i] * n / rank)
        adj_ranked[i] = val
        prev = val

    adjusted = np.empty(n, dtype=float)
    adjusted[order] = np.clip(adj_ranked, 0.0, 1.0)
    return adjusted


def wilcoxon_deg(x_target: np.ndarray, x_other: np.ndarray, gene_names: np.ndarray) -> pd.DataFrame:
    mean_target = x_target.mean(axis=0)
    mean_other = x_other.mean(axis=0)
    logfc = np.log2((mean_target + 1e-9) / (mean_other + 1e-9))

    pvals = np.ones(x_target.shape[1], dtype=float)
    scores = np.zeros(x_target.shape[1], dtype=float)

    for j in range(x_target.shape[1]):
        a = x_target[:, j]
        b = x_other[:, j]

        if np.allclose(a, a[0]) and np.allclose(b, b[0]) and np.isclose(a[0], b[0]):
            pvals[j] = 1.0
            scores[j] = 0.0
            continue

        res = stats.mannwhitneyu(a, b, alternative='two-sided', method='asymptotic')
        pvals[j] = res.pvalue
        scores[j] = res.statistic

    pvals_adj = bh_fdr(pvals)
    df = pd.DataFrame({
        'names': gene_names,
        'scores': scores,
        'logfoldchanges': logfc,
        'pvals': pvals,
        'pvals_adj': pvals_adj,
    })
    return df.sort_values(['pvals_adj', 'pvals', 'logfoldchanges'], ascending=[True, True, False]).reset_index(drop=True)

Set paths and runtime configuration for this interpolation case.

In [ ]:
repo_root = find_repo_root(Path.cwd())
tutorials_root = repo_root / 'docs' / 'tutorials'
section_dir = tutorials_root
data_dir = repo_root / 'data' / 'interpolation'
checkpoints_dir = repo_root / 'checkpoints' / 'interpolation'
resources_dir = tutorials_root / 'resources' / 'interpolation'
case_dir = resources_dir / 'case_Myofibroblasts'

FULL_DATA = data_dir / 'mouse_embryogenesis_aggregated_full_hvg_4000.h5ad'
CKPT = checkpoints_dir / 'myofibroblast_imputation_checkpoint_round6.pth'
MSIGDB_PATH = data_dir / 'msigdb_mouse_v2025_1.json'
CT_TO_TRAJ_JSON = data_dir / 'ct_to_trajectory.json'

PRED_DIR = tutorials_root / 'outputs' / 'interpolation_myofibroblasts' / '00_model_inference'
DEG_DIR = resources_dir / 'deg_results_full'
NOTEBOOK_OUT = tutorials_root / 'outputs' / 'interpolation_myofibroblasts'
TABLE_DIR = NOTEBOOK_OUT / '01_tables'
FIG_DIR = NOTEBOOK_OUT / '02_figures'

for p in [PRED_DIR, DEG_DIR, TABLE_DIR, FIG_DIR, case_dir]:
    p.mkdir(parents=True, exist_ok=True)

CELL_TYPE = 'Myofibroblasts'
TARGET_DAY = 18.25
START_DAY = 8.5
STEP = 0.25
SOURCE_INDICES = list(range(22, 42))

RUN_INFERENCE = True
SKIP_EXISTING_INFERENCE = True
KNN_TRAIN_MAX_CELLS = 80000
N_NEIGHBORS = 20

RUN_DEG = True
SKIP_EXISTING_DEG = True
MIN_CELLS_PER_GROUP = 3

print('repo_root :', '.')
print('tutorials :', tutorials_root.relative_to(repo_root))
print('case_dir  :', case_dir.relative_to(repo_root))
print('full_data :', FULL_DATA.relative_to(repo_root))
print('checkpoint:', CKPT.relative_to(repo_root))
print('pred_dir  :', PRED_DIR.relative_to(repo_root))
print('deg_dir   :', DEG_DIR.relative_to(repo_root))
print('out_root  :', NOTEBOOK_OUT.relative_to(repo_root))


## Step 1: Run temporal interpolation inference for the muscle trajectory
We reproduce adjacent-time prediction files `pred_t{source}_to_t{source+1}.h5ad` with `checkpoint-6`, then use those interpolation outputs as the denoised populations for the the myofibroblast interpolation and denoising case study readout.

In [ ]:
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from navigo.model import MLPTimeGRN, Navigo


def create_model(device):
    return MLPTimeGRN(input_dim=7804, hidden_1=5012, hidden_2=5012).to(device)


def load_and_preprocess_data(input_data_path):
    adata_backed = anndata.read_h5ad(input_data_path, backed='r')
    obs_all = adata_backed.obs.copy()
    obs_all['day_num'] = [float(str(i)[1:]) for i in obs_all['day']]
    all_day = np.sort(obs_all['day_num'].unique())
    time_seq = list(range(len(all_day)))
    map_dict = dict(zip(all_day, time_seq))
    obs_all['time'] = obs_all['day_num'].map(map_dict)

    adata = adata_backed.to_memory()
    adata.obs['day_num'] = obs_all['day_num'].values
    adata.obs['time'] = obs_all['time'].values

    adata_m = np.concatenate([to_dense(adata.layers['Ms']), to_dense(adata.layers['Mu'])], axis=1)
    data = torch.tensor(adata_m, dtype=torch.float32)
    data = (data - data.min(axis=0).values) / (data.max(axis=0).values - data.min(axis=0).values + 1e-7)
    time_label = torch.tensor(adata.obs['time'].values, dtype=torch.float32)
    return data, time_label, adata, map_dict


set_seed(42)

if not FULL_DATA.exists():
    raise FileNotFoundError(FULL_DATA)
if not CKPT.exists():
    raise FileNotFoundError(CKPT)

expected_files = [PRED_DIR / f'pred_t{i}_to_t{i+1}.h5ad' for i in SOURCE_INDICES]
existing_count = sum(int(p.exists()) for p in expected_files)
print(f'Existing prediction files: {existing_count}/{len(expected_files)}')

if RUN_INFERENCE:
    if SKIP_EXISTING_INFERENCE and existing_count == len(expected_files):
        print('All expected prediction files already exist; skipping inference rerun.')
    else:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = create_model(device)
        model.load_state_dict(torch.load(CKPT, map_location=device))
        model.eval()

        flow = Navigo(model=model, num_steps=10, device=device)

        data, time_label, adata, _ = load_and_preprocess_data(FULL_DATA)
        print(f'Loaded data for inference: {adata.n_obs} cells x {adata.n_vars} genes')

        X_train_full = (data[:, :adata.n_vars] + data[:, adata.n_vars:]).cpu().numpy()
        y_train = adata.obs['cell_type'].values

        if KNN_TRAIN_MAX_CELLS and len(y_train) > KNN_TRAIN_MAX_CELLS:
            rng = np.random.default_rng(42)
            keep_idx = rng.choice(len(y_train), size=KNN_TRAIN_MAX_CELLS, replace=False)
            X_train = X_train_full[keep_idx]
            y_train_fit = y_train[keep_idx]
            print(f'KNN training subset: {len(y_train_fit)}/{len(y_train)}')
        else:
            X_train = X_train_full
            y_train_fit = y_train

        knn = KNeighborsClassifier(n_neighbors=N_NEIGHBORS)
        knn.fit(X_train, y_train_fit)

        all_time_set = set(float(x) for x in np.sort(np.unique(time_label.numpy())).tolist())
        for i in SOURCE_INDICES:
            source_time = float(i)
            target_time = float(i + 1)
            out_file = PRED_DIR / f'pred_t{i}_to_t{i+1}.h5ad'

            if SKIP_EXISTING_INFERENCE and out_file.exists():
                continue
            if source_time not in all_time_set or target_time not in all_time_set:
                print(f'[SKIP] missing source/target time for {i}->{i+1}')
                continue

            source_mask = time_label == source_time
            z0 = data[source_mask].to(device)
            t0 = torch.tensor([source_time] * z0.shape[0]).to(device)
            t1 = torch.tensor([target_time] * z0.shape[0]).to(device)

            pred_forward = flow.sample_ode_time_interval(z_full=z0, t_start=t0, t_end=t1, N=100)
            pred_forward = pred_forward[:, :adata.n_vars] + pred_forward[:, adata.n_vars:]
            y_pred = knn.predict(pred_forward)

            pred_adata = anndata.AnnData(X=pred_forward)
            pred_adata.obs['predicted_cell_type'] = y_pred
            pred_adata.obs['source_time'] = source_time
            pred_adata.obs['target_time'] = target_time
            pred_adata.write_h5ad(out_file)
            print('[OK]', out_file.name)

post_count = sum(int(p.exists()) for p in expected_files)
print(f'Prediction availability after step: {post_count}/{len(expected_files)}')


## Step 2: Compute day-specific DEG tables for Myofibroblasts
For each developmental day in Myofibroblasts, we compare the focal day against all other days using the Mann-Whitney test with Benjamini-Hochberg correction. These DEG tables support the marker and pathway summaries used in the denoising analysis.

In [ ]:
def normalize_to_x(adata: anndata.AnnData) -> anndata.AnnData:
    adata_m = np.concatenate([to_dense(adata.layers['Ms']), to_dense(adata.layers['Mu'])], axis=1)
    data = (adata_m - adata_m.min(axis=0)) / (adata_m.max(axis=0) - adata_m.min(axis=0) + 1e-7)
    n_vars = adata.n_vars
    adata.X = data[:, :n_vars] + data[:, n_vars:]
    return adata


def sorted_days(day_values):
    days = sorted(pd.Series(day_values).astype(str).unique())

    def day_key(day):
        if day.startswith('E'):
            return float(day[1:])
        if day == 'P0':
            return 19.5
        return float('inf')

    return sorted(days, key=day_key)


def sanitize_cell_type(cell_type: str) -> str:
    return cell_type.replace('/', '_').replace(' ', '_').replace('(', '|').replace(')', '|')


ct_file = sanitize_cell_type(CELL_TYPE)
expected_deg_days = sorted([float(str(i)[1:]) for i in anndata.read_h5ad(FULL_DATA, backed='r').obs['day'].unique() if str(i).startswith('E')])
existing_deg = sorted(DEG_DIR.glob(f'{ct_file}_E*_deg.csv'))
print(f'Existing DEG files: {len(existing_deg)}')

if RUN_DEG:
    # subset first for efficiency
    adata_backed = anndata.read_h5ad(FULL_DATA, backed='r')
    obs = adata_backed.obs.copy()
    mask = (obs['cell_type'].astype(str) == CELL_TYPE).values
    adata_ct = adata_backed[mask].to_memory()
    adata_ct = normalize_to_x(adata_ct)

    x_ct = to_dense(adata_ct.X)
    gene_names = np.asarray(adata_ct.var_names.astype(str))
    days = sorted_days(adata_ct.obs['day'])
    day_values = adata_ct.obs['day'].astype(str).values

    print(f'Loaded CT subset: {adata_ct.n_obs} cells, {adata_ct.n_vars} genes, {len(days)} days')
    for day in days:
        out_file = DEG_DIR / f'{ct_file}_{day.replace("P0", "E19.5")}_deg.csv'
        if SKIP_EXISTING_DEG and out_file.exists():
            continue

        day_mask = day_values == day
        other_mask = ~day_mask
        n_target = int(day_mask.sum())
        n_other = int(other_mask.sum())
        if n_target < MIN_CELLS_PER_GROUP or n_other < MIN_CELLS_PER_GROUP:
            print(f'[SKIP] {day}: target={n_target}, other={n_other}')
            continue

        df_deg = wilcoxon_deg(x_ct[day_mask], x_ct[other_mask], gene_names)
        df_deg.to_csv(out_file, index=False)
        print('[OK]', out_file.name)

existing_deg_after = sorted(DEG_DIR.glob(f'{ct_file}_E*_deg.csv'))
print(f'DEG files after step: {len(existing_deg_after)}')

## Step 3: Run paper-style summary scripts
This reproduces the proportion, marker-gene, and pathway-enrichment tables used for the myofibroblast denoising interpretation.

In [ ]:
point = int(round((TARGET_DAY - START_DAY) / STEP))
prop_csv = TABLE_DIR / '01_trajectory_proportion_table.csv'

analysis_cmds = [
    [
        sys.executable,
        str(case_dir / 'panel_marker_gene.py'),
        '--input_data', str(FULL_DATA),
        '--pred_dir', str(PRED_DIR),
        '--deg_dir', str(DEG_DIR),
        '--cell_type', CELL_TYPE,
        '--target_day', str(TARGET_DAY),
    ],
    [
        sys.executable,
        str(case_dir / 'panel_pathway_enrichment.py'),
        '--input_data', str(FULL_DATA),
        '--pred_dir', str(PRED_DIR),
        '--deg_dir', str(DEG_DIR),
        '--msigdb_path', str(MSIGDB_PATH),
        '--cell_type', CELL_TYPE,
        '--target_day', str(TARGET_DAY),
    ],
    [
        sys.executable,
        str(case_dir / 'panel_proportion_count.py'),
        '--pred_dir', str(PRED_DIR),
        '--ct_to_trajectory_json', str(CT_TO_TRAJ_JSON),
        '--cell_type', CELL_TYPE,
        '--target_day', str(TARGET_DAY),
        '--output_csv', str(prop_csv),
    ],
]

for cmd in analysis_cmds:
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

marker_csv = case_dir / f'{ct_file}_marker_genes_t{point}.csv'
shared_csv = case_dir / f'{ct_file}_shared_pathways.csv'
for path in [marker_csv, shared_csv, prop_csv]:
    if not path.exists():
        raise FileNotFoundError(path)

pd.read_csv(prop_csv)

## Step 4: Materialize summary views for inline review
Render the myofibroblast proportion trajectory, marker-gene dynamics, and pathway enrichment comparison into a single notebook output workspace.

In [ ]:
if str(resources_dir) not in sys.path:
    sys.path.insert(0, str(resources_dir))

from render_end_to_end_figures import render_all_panels

summary = render_all_panels(
    input_data=FULL_DATA,
    pred_dir=PRED_DIR,
    ct_to_trajectory_json=CT_TO_TRAJ_JSON,
    cell_type=CELL_TYPE,
    target_day=TARGET_DAY,
    start_day=START_DAY,
    step=STEP,
    case_dir=case_dir,
    output_root=NOTEBOOK_OUT,
)

for key, path in summary['paths'].items():
    print(f'{key}: {Path(path).relative_to(repo_root)}')
if summary['missing_display_genes']:
    print('Missing display genes:', summary['missing_display_genes'])

j_fig = Path(summary['paths']['panel_j'])
k_fig = Path(summary['paths']['panel_k'])
l_fig = Path(summary['paths']['panel_l'])



## Biological interpretation guide
- **Proportion continuity**: Compare noisy and Navigo-derived proportions across the muscle trajectory; the key question is whether the apparent E18.25 irregularity in myofibroblast abundance is pulled back toward the trend suggested by adjacent stages.
- **Marker continuity**: Marker continuity is evaluated with `Cdkn1c`, `Synpo2`, `Myf6`, `Tnnc2`, `Myog`, and `Aldoa`; improved agreement with neighboring stages supports successful denoising.
- **Pathway recovery**: Pathway analysis tests whether the denoised population recovers coherent enrichment for skeletal muscle, metabolic, stress-response, and movement-associated programs that are weakened or distorted in the noisy E18.25 population.


## Outputs
The notebook writes summary tables under `01_tables/` and rendered views under `02_figures/`.